# U-Values and Thermal Calculations

This notebook calculates thermal conductivity (lambda) values, R-values, and U-values for building materials, with a focus on External Wall Insulation (EWI) payback analysis.

## Setup: Import Libraries

In [1]:
import pint

ureg = pint.UnitRegistry()
ureg.define('gbp = [currency]') 
ureg.gbp.symbol = '£'


## Material Properties


In [13]:
LAMBDA = {    
    "xps": 0.04,  # Polyisocyanurate or extruded polystyrene    
    "brick": 0.77,  # Medium density clay brick (~1800 kg/m³)
    "clay_brick_lightweight": 0.5,  # Lightweight clay brick (< 1700 kg/m³)    
    "plasterboard": 0.25,  # Standard plasterboard (gypsum)
    "plaster": 0.77 # assumed
    # add a few more here when everything else works
}

## formulae for R and U

In [14]:
def lambda_val(material):
    """Get thermal conductivity with units."""
    return LAMBDA[material] * ureg.W / (ureg.m * ureg.kelvin) 

def R_value(element, thickness):
    """Calculate R-value (thermal resistance) for a wall element."""
    return  thickness/ lambda_val(element)

def U_value(wall):
    """
    The U value of the wall, in SI units.
    """
    units_of_U = ureg.watt / ureg.meter ** 2 / ureg.K
    tot_R = 0.0 / units_of_U
    for i in wall:
        R = R_value(i,wall[i] * ureg.meters)
        tot_R += R
    return 1./tot_R
    
assert lambda_val('plaster') == 0.77 * ureg.W / ureg.m / ureg.K  # check 

## Get U values for wall with and without EWI/IWI

In [6]:
# use a dict type to represent the elements in the wall.
u_with_EWI = U_value(
    {
    "plasterboard": 0.013,  # Internal plaster finish
    "brick": 0.220,  # Main structural brick layer
    "xps": 0.0700,})  # Internal insulation board

print(f"u_with_EWI: {u_with_EWI:.2fP~}")

u_with_EWI: 0.48 W/K/m²


In [7]:

u_without_EWI = U_value(
    {
    "plaster": 0.025,  # Internal plaster finish
    "brick": 0.2200,  # Main structural brick layer
    }) 

print(f"u_without_EWI: {u_without_EWI:.2fP~}")

u_without_EWI: 3.14 W/K/m²


## Compute actual energy losses for wall before and after

### Calculate R-values for EWI Components

### Energy and Cost Calculations

In [15]:
# Energy depends on integral of temperature shortfall over the year.
# this is HDD downloaded from https://www.degreedays.net/ for Luton Airport for a base 
# of 15.5°C (traditional base temp)
degree_days = 2009.3 * ureg.K * 60*60*24 * ureg.s  
print(f"degree days (in SI units!) {degree_days:.2eP~}")

degree days (in SI units!) 1.61×10⁸ K·s


In [16]:
# gas price, as quoted on your gas bill:
gas_price = 0.03*ureg.gbp / ((1000 * ureg.W) * (60*60*ureg.s))   # gbp/kWh (2021 pre-crisis price)
print(f"gas price {gas_price:.2eP~}")

gas price 8.33×10⁻⁹ gbp/s/W


In [17]:
# Work out annual cost

# base temperate should be about 15C.
# degree days should be about 2500-600 (if base temp is 15C).

# Calculate U-value improvement
delta_u = u_without_EWI - u_with_EWI

print(f"Change in U-value from EWI: {delta_u:.4fP~}")

Change in U-value from EWI: 2.6639 W/K/m²


In [20]:
# Calculate unit savings 
saving = gas_price * delta_u # per unit time
print(f"Unit saving (i.e. per second): {saving:.6eP~}")
saving_pd = saving*60*60*24*ureg.s
print(f"saving per day: {saving_pd:.2eP~}")

year_cost_psm = degree_days  * gas_price * delta_u

print(f"Annual cost saving: {year_cost_psm:.2fP~}")

Unit saving (i.e. per second): 2.219887×10⁻⁸ gbp/K/m²/s
saving per day: 1.92×10⁻³ gbp/K/m²
Annual cost saving: 3.58 gbp/m²


### Payback Calculation

In [22]:
# Installation cost
# Based on non-binding verbal quote from TAW: gbp30K for front and side elevations (~150m²)
installation_cost = 200 * ureg.gbp / (ureg.meter**2)  # gbp/m²

print(f"Installation cost: {installation_cost:.0fP~}")
print(f"Annual saving per m²: {year_cost_psm:.2fP~}")

simple_payback = installation_cost / year_cost_psm
print(f"Simple payback period: {simple_payback:.1f} years")


Installation cost: 200 gbp/m²
Annual saving per m²: 3.58 gbp/m²
Simple payback period: 55.8 dimensionless years
